<div align="center">

# NB2 · The XAI Toolkit I: Tabular Clinical Data

**Explainable Artificial Intelligence in Medical Systems: From Black Box Models to Reliable Clinical Decision Support**<br>
IEEE CARS 2026 Workshop · 28 October 2026 · Session 10:20 to 10:45

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/utkukose/xai-cars2026-NB-workshop/blob/main/notebooks/NB2_XAI_Toolkit_Tabular.ipynb)

Prof. Dr. Utku Köse · Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

</div>

## About this notebook

This notebook is a guided tour of more than twenty explanation methods, organised by the question each one answers. All methods are applied to the same clinical problem and, where possible, to the same model, so that their answers can be compared directly. The tour ends with the part that is most often skipped in practice: measuring whether an explanation can be trusted.

The data are the Cleveland heart disease records from the UCI Machine Learning Repository [1, 2]: 303 patients referred for coronary angiography, 13 clinical variables, and the angiographic diagnosis. The file is downloaded directly from UCI when the notebook runs. A full run takes about three minutes on a free Colab CPU.

| Question | Methods in this notebook |
|---|---|
| Can the model itself be read? | logistic regression, decision tree, generalised additive model (GAM), explainable boosting machine (EBM) |
| What matters for the model overall? | permutation importance, SHAP summary, partial dependence (PDP) and ICE curves, accumulated local effects (ALE), SAGE |
| Why this prediction for this patient? | SHAP waterfall, LIME, MAPLE, integrated gradients, occlusion |
| What would change the prediction? | counterfactual explanation, Anchors rule, global surrogate tree |
| What does the geometry of the model say? | GEMEX geodesic sensitivity |
| Can the explanation be trusted? | faithfulness (deletion), monotonicity, stability, disagreement between methods |

**Companion tools.** The *Explain a patient* and *Shapley by hand* tabs of the [XAI Lab](https://utkukose.github.io/xai-cars2026-NB-workshop/) run a model of the same kind in the browser, with exact Shapley values that update as the patient's values are edited.

In [ ]:
import importlib.util, subprocess, sys
PACKAGES = {"shap": "shap>=0.46", "lime": "lime==0.2.0.1", "gemex": "gemex>=1.2.2", "interpret": "interpret-core>=0.6"}
missing = [spec for module, spec in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
import os, io, sys, json, time, zipfile, hashlib, warnings, urllib.request, urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUICK = os.environ.get("XAI_WORKSHOP_QUICK", "0") == "1"   # smaller runs for automated checks
SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("workshop_data")
DATA_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
trapezoid = getattr(np, "trapezoid", None) or np.trapz

INK, ALERT, SAFE, MUTED, AMBER, VIOLET = "#1D2742", "#C0392B", "#1F7A6D", "#8A93A6", "#B26B00", "#6C4AB6"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
})


def fetch(urls, dest, timeout=120):
    """Download the first reachable URL into dest, reuse an existing copy, and return the path."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in ([urls] if isinstance(urls, str) else urls):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            part = dest.with_name(dest.name + ".part")
            part.write_bytes(payload)
            part.replace(dest)
            return dest
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("No source could be reached:\n" + "\n".join(errors))


print(f"numpy {np.__version__} | pandas {pd.__version__} | quick mode: {QUICK}")

In [ ]:
import shap
import ipywidgets as widgets
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
from sklearn.exceptions import ConvergenceWarning
from interpret.glassbox import ExplainableBoostingClassifier
from lime.lime_tabular import LimeTabularExplainer
from gemex import Explainer

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## 1. Data

The processed Cleveland file has 303 rows; six have a missing value in the number of major vessels or the thallium scan and are removed, as in most published analyses. The target is the presence of angiographic disease (any narrowing greater than 50 percent). Variable names are expanded for readability.

In [ ]:
UCI_FILE = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
UCI_ZIP = "https://archive.ics.uci.edu/static/public/45/heart+disease.zip"
RAW = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
RENAME = {"sex": "sex_male", "cp": "chest_pain_type", "trestbps": "resting_bp", "chol": "cholesterol",
          "fbs": "fasting_glucose_high", "restecg": "resting_ecg", "thalach": "max_heart_rate",
          "exang": "exercise_angina", "oldpeak": "st_depression", "slope": "st_slope",
          "ca": "major_vessels", "thal": "thallium_scan"}


def load_cleveland():
    path = DATA_DIR / "processed.cleveland.data"
    if not path.exists():
        try:
            fetch([UCI_FILE], path)
        except RuntimeError:
            with zipfile.ZipFile(fetch([UCI_ZIP], DATA_DIR / "heart+disease.zip")) as z:
                path.write_bytes(z.read("processed.cleveland.data"))
    return pd.read_csv(path, header=None, names=RAW, na_values="?")


heart = load_cleveland().dropna().rename(columns=RENAME).reset_index(drop=True)
y = (heart.pop("num") > 0).astype(int).rename("disease")
X = heart.astype(float)
FEATURES = list(X.columns)
CONT = ["age", "resting_bp", "cholesterol", "max_heart_rate", "st_depression"]
CAT = ["chest_pain_type", "resting_ecg", "st_slope", "thallium_scan"]
BIN = ["sex_male", "fasting_glucose_high", "exercise_angina"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(f"{len(X)} patients ({y.mean():.0%} with disease) · train {len(X_train)} · test {len(X_test)}")
X.describe().T[["mean", "min", "max"]].round(1)

Coding of the categorical variables: chest pain type 1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic; resting ECG 0 normal, 1 ST-T abnormality, 2 left ventricular hypertrophy; ST slope 1 upsloping, 2 flat, 3 downsloping; thallium scan 3 normal, 6 fixed defect, 7 reversible defect; major vessels 0 to 3 coloured by fluoroscopy.

## 2. Glass-box models: when the model is the explanation

An interpretable model can often reach the accuracy of a black box on tabular clinical data [3]. Four such models are fitted below. Logistic regression is read through odds ratios. A shallow decision tree is read as a set of rules. A generalised additive model (GAM) replaces each linear term with a smooth curve [4]; here the curves are built from cubic splines. The explainable boosting machine (EBM) is a GAM learned by boosting, with a small number of pairwise interaction terms [5, 6].

In [ ]:
lr_model = make_pipeline(
    ColumnTransformer([("num", StandardScaler(), CONT + ["major_vessels"]),
                       ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
                       ("bin", "passthrough", BIN)]),
    LogisticRegression(C=0.5, max_iter=5000))
lr_model.fit(X_train, y_train)
names = lr_model[0].get_feature_names_out()
odds = pd.DataFrame({"coefficient": lr_model[-1].coef_[0], "odds ratio": np.exp(lr_model[-1].coef_[0])},
                    index=[n.split("__")[1] for n in names])
odds.reindex(odds["coefficient"].abs().sort_values(ascending=False).index).head(10).round(2)

For standardised continuous variables the odds ratio refers to one standard deviation; for indicator columns it compares the category with the others.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, random_state=SEED).fit(X_train, y_train)
print(export_text(tree, feature_names=FEATURES, decimals=1))

In [ ]:
gam_prep = ColumnTransformer(
    [(f"s_{c}", SplineTransformer(n_knots=5, degree=3), [c]) for c in CONT]
    + [("cat", OneHotEncoder(handle_unknown="ignore"), CAT), ("rest", "passthrough", BIN + ["major_vessels"])])
gam = make_pipeline(gam_prep, LogisticRegression(C=0.3, max_iter=5000)).fit(X_train, y_train)
gam_coef = gam[-1].coef_[0]


def gam_shape(col):
    grid = np.linspace(X[col].quantile(0.02), X[col].quantile(0.98), 120)
    spline = gam[0].named_transformers_[f"s_{col}"]
    cols = gam[0].output_indices_[f"s_{col}"]
    curve = spline.transform(pd.DataFrame({col: grid})) @ gam_coef[cols]
    centre = (spline.transform(X_train[[col]]) @ gam_coef[cols]).mean()
    return grid, curve - centre


fig, axes = plt.subplots(1, len(CONT), figsize=(15, 2.8), sharey=True)
for ax, col in zip(axes, CONT):
    g, s = gam_shape(col)
    ax.plot(g, s, color=ALERT, lw=2.2)
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_title(col)
axes[0].set_ylabel("contribution to log-odds")
fig.suptitle("GAM shape functions: each curve is the whole story for its variable", y=1.04)
plt.show()

In [ ]:
ebm = ExplainableBoostingClassifier(interactions=3, random_state=SEED).fit(X_train, y_train)
ebm_importance = pd.Series(ebm.term_importances(), index=ebm.term_names_).sort_values()
ax = ebm_importance.plot.barh(figsize=(6.5, 4.2), color=INK)
ax.set_xlabel("mean absolute contribution (log-odds)")
ax.set_title("EBM term importances, including learned interactions")
plt.tight_layout()
plt.show()

The black box used for the rest of the notebook is a gradient boosting ensemble of 150 trees. The comparison below shows how much accuracy, if any, the interpretable models give up on this dataset.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.9,
                                random_state=SEED).fit(X_train, y_train)
models = {"logistic regression": lr_model, "decision tree (depth 3)": tree, "GAM (splines)": gam,
          "EBM": ebm, "gradient boosting (black box)": gb}
pd.DataFrame({name: {"test AUROC": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]),
                     "test accuracy": accuracy_score(y_test, m.predict(X_test))}
              for name, m in models.items()}).T.round(3)

## 3. Global attribution: what matters for the model overall

**Permutation importance** measures the loss in test AUROC when one variable is shuffled [7]. **SHAP** assigns each prediction a fair share of credit per feature, following the Shapley value from cooperative game theory [8, 9]; the exact tree algorithm makes this fast for boosted trees [10]. Here SHAP values are computed on the probability scale with a background sample of 100 training patients, so the values of one patient add up to the difference between that patient's predicted risk and the average risk.

In [ ]:
pi = permutation_importance(gb, X_test, y_test, scoring="roc_auc", n_repeats=10 if QUICK else 30, random_state=SEED)
background = X_train.sample(100, random_state=SEED)
explainer = shap.TreeExplainer(gb, data=background, feature_perturbation="interventional", model_output="probability")
sv = explainer(X_test)

fig, ax = plt.subplots(figsize=(6, 4.4))
order = np.argsort(pi.importances_mean)
ax.barh(np.array(FEATURES)[order], pi.importances_mean[order], xerr=pi.importances_std[order], color=INK)
ax.set_xlabel("drop in test AUROC when shuffled")
ax.set_title("Permutation importance")
plt.tight_layout()
plt.show()
shap.plots.beeswarm(sv, max_display=13, show=False)
plt.title("SHAP summary: one dot per test patient")
plt.show()

**Partial dependence** shows the average predicted risk as one variable moves across its range, and **individual conditional expectation** (ICE) curves show the same for single patients [11, 12]. Partial dependence can mislead when variables are correlated, because it evaluates the model on combinations that never occur, such as a 75-year-old with the maximum heart rate of a 30-year-old. **Accumulated local effects** (ALE) avoid this by averaging local changes within narrow bins of real data [13]. Age and maximum heart rate are correlated in this dataset, which makes age a good test case.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
PartialDependenceDisplay.from_estimator(gb, X_train, ["max_heart_rate", "st_depression", "age"], kind="both",
                                        subsample=60, random_state=SEED, ax=axes,
                                        ice_lines_kw={"alpha": 0.15, "color": MUTED},
                                        pd_line_kw={"color": ALERT, "lw": 2.5})
plt.tight_layout()
plt.show()


def ale_1d(model, data, feature, n_bins=10):
    """First-order accumulated local effects on the probability scale."""
    edges = np.unique(np.quantile(data[feature], np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, data[feature].to_numpy(), side="right") - 1, 0, len(edges) - 2)
    effect, count = np.zeros(len(edges) - 1), np.zeros(len(edges) - 1)
    for k in range(len(edges) - 1):
        rows = data[idx == k]
        if len(rows) == 0:
            continue
        lo, hi = rows.copy(), rows.copy()
        lo[feature], hi[feature] = edges[k], edges[k + 1]
        effect[k] = (model.predict_proba(hi)[:, 1] - model.predict_proba(lo)[:, 1]).mean()
        count[k] = len(rows)
    ale = np.concatenate([[0.0], np.cumsum(effect)])
    ale -= np.average(0.5 * (ale[:-1] + ale[1:]), weights=np.maximum(count, 1e-12))
    return edges, ale


print("correlation between age and maximum heart rate:", round(X["age"].corr(X["max_heart_rate"]), 2))
fig, ax = plt.subplots(figsize=(5.5, 3.6))
edges, ale = ale_1d(gb, X_train, "age")
grid = np.linspace(X_train["age"].min(), X_train["age"].max(), 40)
pdp = [gb.predict_proba(X_train.assign(age=v))[:, 1].mean() for v in grid]
ax.plot(grid, np.array(pdp) - np.mean(pdp), color=MUTED, lw=2, label="PDP (centred)")
ax.plot(edges, ale, color=ALERT, lw=2.4, marker="o", ms=3, label="ALE")
ax.set_xlabel("age")
ax.set_ylabel("effect on predicted risk")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**SAGE** asks a different global question: how much does each variable reduce the model's loss when it is known [14]? It is the Shapley value of the prediction task rather than of a single prediction. The estimator below samples permutations of the variables, adds them one at a time, and replaces unknown variables with values drawn from the background data.

In [ ]:
def sage_values(model, X_eval, y_eval, X_bg, n_perm=30, seed=SEED):
    rs = np.random.default_rng(seed)
    Xe, Xb = X_eval.to_numpy(), X_bg.to_numpy()
    n, d, k = len(Xe), Xe.shape[1], len(Xb)
    rep, tile = np.repeat(Xe, k, axis=0), np.tile(Xb, (n, 1))

    def loss(known):
        mixed = np.where(known, rep, tile)
        p = model.predict_proba(pd.DataFrame(mixed, columns=X_eval.columns))[:, 1].reshape(n, k).mean(axis=1)
        return log_loss(y_eval, p, labels=[0, 1])

    phi, empty = np.zeros(d), loss(np.zeros(d, dtype=bool))
    for _ in range(n_perm):
        known, prev = np.zeros(d, dtype=bool), empty
        for j in rs.permutation(d):
            known[j] = True
            cur = loss(known)
            phi[j] += prev - cur
            prev = cur
    return pd.Series(phi / n_perm, index=X_eval.columns)


sage = sage_values(gb, X_test, y_test, background.iloc[:24], n_perm=8 if QUICK else 30)
pd.DataFrame({"SAGE (loss reduction)": sage, "permutation importance": pi.importances_mean},
             index=FEATURES).sort_values("SAGE (loss reduction)", ascending=False).round(4)

## 4. Local explanations: why this prediction for this patient?

The patient below is a test patient with angiographic disease and a predicted risk close to 0.75. Each method answers the same question for this patient in its own way.

In [ ]:
proba_test = gb.predict_proba(X_test)[:, 1]
diseased = np.where(y_test.to_numpy() == 1)[0]
i_pat = diseased[np.argmin(np.abs(proba_test[diseased] - 0.75))]
x_pat = X_test.iloc[i_pat]
print(f"Test patient {i_pat}: predicted risk {proba_test[i_pat]:.3f}, true diagnosis: disease")
x_pat.to_frame("value").T

**SHAP waterfall.** The explanation starts at the average predicted risk of the background patients and adds one contribution per variable until it reaches this patient's risk.

In [ ]:
shap.plots.waterfall(sv[i_pat], max_display=10, show=False)
plt.title("SHAP explanation for the selected patient")
plt.show()

**LIME** fits a weighted linear model to the black box's answers on random perturbations around the patient [15]. Because the perturbations are random, two runs can disagree. The cell below runs LIME five times with different seeds and reports how often the top five variables stay the same.

In [ ]:
cat_idx = [FEATURES.index(c) for c in CAT + BIN]


def lime_attribution(x_row, seed, num_samples=2000):
    le = LimeTabularExplainer(X_train.to_numpy(), feature_names=FEATURES, class_names=["no disease", "disease"],
                              categorical_features=cat_idx, discretize_continuous=True, mode="classification",
                              random_state=seed)
    exp = le.explain_instance(np.asarray(x_row, dtype=float), gb.predict_proba,
                              num_features=len(FEATURES), num_samples=num_samples)
    w = np.zeros(len(FEATURES))
    for j, weight in exp.as_map()[1]:
        w[j] = weight
    return w


lime_runs = np.array([lime_attribution(x_pat, seed) for seed in range(5)])
top5 = [set(np.argsort(-np.abs(w))[:5]) for w in lime_runs]
jaccard = np.mean([len(a & b) / len(a | b) for i, a in enumerate(top5) for b in top5[i + 1:]])
fig, ax = plt.subplots(figsize=(7.5, 4))
for k, w in enumerate(lime_runs):
    ax.scatter(w, np.arange(len(FEATURES)) + (k - 2) * 0.1, s=18, color=VIOLET, alpha=0.7)
ax.set_yticks(range(len(FEATURES)), FEATURES)
ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel("LIME weight")
ax.set_title(f"Five LIME runs for one patient · mean top-5 overlap (Jaccard) {jaccard:.2f}")
plt.tight_layout()
plt.show()

**MAPLE** replaces LIME's random neighbourhood with a supervised one [16]. A random forest is trained to imitate the black box; training patients that land in the same leaves as the explained patient receive high weight, and a weighted linear model is fitted on them. The contribution of a variable is its local coefficient times the patient's standardised value.

In [ ]:
mu, sd = X_train.mean(), X_train.std()
gb_train = gb.predict_proba(X_train)[:, 1]
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=SEED).fit(X_train, gb_train)
leaves_train = forest.apply(X_train)


def maple_attribution(x_row):
    same = leaves_train == forest.apply(x_row.to_frame().T)[0]
    weights = (same / same.sum(axis=0, keepdims=True)).mean(axis=1)
    local = Ridge(alpha=1.0).fit((X_train - mu) / sd, gb_train, sample_weight=weights)
    return local.coef_ * ((x_row - mu) / sd).to_numpy()


maple_pat = maple_attribution(x_pat)
pd.Series(maple_pat, index=FEATURES).sort_values(key=np.abs, ascending=False).head(8).round(3).to_frame("MAPLE")

**Integrated gradients** need a differentiable model [17]. A small neural network is trained on the standardised variables, and its gradient is integrated along the straight path from a baseline (the average patient) to the explained patient. The method satisfies completeness: the attributions add up to the difference between the two log-odds, up to the error of the numerical integral, which the cell verifies.

In [ ]:
scaler = StandardScaler().fit(X_train)
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3, max_iter=3000, random_state=SEED)
mlp.fit(scaler.transform(X_train), y_train)
print("neural network test AUROC:", round(roc_auc_score(y_test, mlp.predict_proba(scaler.transform(X_test))[:, 1]), 3))


def mlp_logit_and_gradient(model, Z):
    """Log-odds of a ReLU MLPClassifier and its gradient with respect to the inputs."""
    h, pre = Z, []
    for W, b in zip(model.coefs_[:-1], model.intercepts_[:-1]):
        z = h @ W + b
        pre.append(z)
        h = np.maximum(z, 0.0)
    logit = (h @ model.coefs_[-1] + model.intercepts_[-1]).ravel()
    grad = np.repeat(model.coefs_[-1].T, len(Z), axis=0)
    for W, z in zip(reversed(model.coefs_[:-1]), reversed(pre)):
        grad = (grad * (z > 0)) @ W.T
    return logit, grad


def integrated_gradients(model, z, z_base, steps=128):
    alphas = (np.arange(steps) + 0.5) / steps
    path = z_base + alphas[:, None] * (z - z_base)
    _, grads = mlp_logit_and_gradient(model, path)
    return (z - z_base) * grads.mean(axis=0)


z_pat = scaler.transform(x_pat.to_frame().T)[0]
z_base = np.zeros_like(z_pat)
ig = integrated_gradients(mlp, z_pat, z_base)
logits, _ = mlp_logit_and_gradient(mlp, np.vstack([z_pat, z_base]))
print(f"sum of attributions {ig.sum():+.4f} · logit difference {logits[0] - logits[1]:+.4f}")
pd.Series(ig, index=FEATURES).sort_values(key=np.abs, ascending=False).head(8).round(3).to_frame("integrated gradients")

## 5. Contrastive and rule-based explanations

A **counterfactual explanation** states the smallest plausible change that would alter the prediction [18]. Clinical plausibility matters: age and sex are fixed, and the searched values stay within the range seen in the training data. The search below changes one variable at a time, choosing the move that lowers the risk most per unit of normalised change, and then removes any change that turns out to be unnecessary. Diverse and constrained variants exist in the literature [19].

A counterfactual and an adversarial example are close relatives. The first is a plausible change that a patient and clinician can act on; the second is a change an attacker makes so that nobody notices. The attack notebooks return to this link.

In [ ]:
STEPS = {"resting_bp": 5.0, "cholesterol": 10.0, "max_heart_rate": 5.0, "st_depression": 0.2,
         "exercise_angina": None, "major_vessels": None}
LOW, HIGH = X_train.quantile(0.02), X_train.quantile(0.98)
SCALE = X_train.std()


def risk(frame):
    return gb.predict_proba(frame)[:, 1]


def counterfactual(x_row, target=0.5, max_steps=40):
    x = x_row.copy()
    for _ in range(max_steps):
        p_now = risk(x.to_frame().T)[0]
        if p_now < target:
            break
        moves = []
        for f, step in STEPS.items():
            if step is None:
                options = [1.0 - x[f]] if f == "exercise_angina" else [x[f] - 1.0]
            else:
                options = [x[f] - step, x[f] + step]
            for v in options:
                if LOW[f] <= v <= HIGH[f]:
                    moves.append((f, v))
        if not moves:
            break
        cand = pd.DataFrame([x.to_dict()] * len(moves))
        for k, (f, v) in enumerate(moves):
            cand.loc[k, f] = v
        gain = (p_now - risk(cand)) / np.array([abs(v - x[f]) / SCALE[f] for f, v in moves])
        if gain.max() <= 0:
            break
        f, v = moves[int(np.argmax(gain))]
        x[f] = v
    for f in STEPS:
        if x[f] != x_row[f]:
            trial = x.copy()
            trial[f] = x_row[f]
            if risk(trial.to_frame().T)[0] < target:
                x = trial
    return x


cf = counterfactual(x_pat)
changed = [f for f in FEATURES if cf[f] != x_pat[f]]
print(f"risk {risk(x_pat.to_frame().T)[0]:.3f} -> {risk(cf.to_frame().T)[0]:.3f}")
pd.DataFrame({"patient": x_pat[changed], "counterfactual": cf[changed]}).round(1)

When the counterfactual needs only a small change in a single variable, as often happens with tree ensembles, it reveals a sharp step in the model's response. Such a step has no clinical justification, and it is exactly the kind of fragility that an attacker looks for.

**Anchors** explain a prediction with an IF-THEN rule that is sufficient to keep the prediction in place, together with its precision and coverage [20]. The simplified search below adds conditions greedily. Precision is estimated by sampling patients who satisfy the rule and checking how often the model gives the same answer; coverage is the share of training patients to whom the rule applies.

In [ ]:
QUART = {c: np.quantile(X_train[c], [0.25, 0.5, 0.75]) for c in CONT}


def condition(feature, value):
    if feature in CONT:
        edges = np.concatenate([[-np.inf], QUART[feature], [np.inf]])
        k = int(np.clip(np.searchsorted(edges, value, side="left") - 1, 0, 3))
        lo, hi = edges[k], edges[k + 1]
        text = (f"{feature} <= {hi:g}" if np.isinf(lo) else f"{feature} > {lo:g}" if np.isinf(hi)
                else f"{lo:g} < {feature} <= {hi:g}")
        return (lambda s, lo=lo, hi=hi: (s > lo) & (s <= hi)), text
    return (lambda s, v=value: s == v), f"{feature} = {value:g}"


def anchor(x_row, target_precision=0.95, max_len=4, n=600, seed=SEED):
    rs = np.random.default_rng(seed)
    label = int(risk(x_row.to_frame().T)[0] >= 0.5)
    rule, best = [], (0.0, 1.0)
    while len(rule) < max_len and best[0] < target_precision:
        scored = []
        for f in FEATURES:
            if f in [r[0] for r in rule]:
                continue
            trial = rule + [(f, *condition(f, x_row[f]))]
            mask = np.all([test(X_train[g]) for g, test, _ in trial], axis=0)
            if mask.sum() < 3:
                continue
            sample = X_train.sample(n, replace=True, random_state=int(rs.integers(1e9))).reset_index(drop=True)
            pool = X_train[mask]
            for g, _, _ in trial:
                sample[g] = pool[g].sample(n, replace=True, random_state=int(rs.integers(1e9))).to_numpy()
            precision = ((risk(sample) >= 0.5).astype(int) == label).mean()
            scored.append((precision, mask.mean(), trial))
        if not scored:
            break
        precision, coverage, rule = max(scored, key=lambda s: (s[0], s[1]))
        best = (precision, coverage)
    return " AND ".join(t for _, _, t in rule), best


rule_text, (precision, coverage) = anchor(x_pat)
print(f"IF {rule_text}\nTHEN {'disease' if proba_test[i_pat] >= 0.5 else 'no disease'}"
      f"  (precision {precision:.2f}, coverage {coverage:.2f})")

A **global surrogate** trains an interpretable model to imitate the black box. Its fidelity, the agreement with the black box on unseen patients, says how far its rules can be trusted as a description of the black box.

In [ ]:
surrogate = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, gb.predict(X_train))
print(f"fidelity to the black box on test patients: {accuracy_score(gb.predict(X_test), surrogate.predict(X_test)):.2f}")
fig, ax = plt.subplots(figsize=(13, 4.6))
plot_tree(surrogate, feature_names=FEATURES, class_names=["no disease", "disease"], filled=True,
          impurity=False, proportion=True, fontsize=7, ax=ax)
plt.show()

## 6. Geometric XAI: GEMEX

GEMEX, Geodesic Entropic Manifold Explainability, treats the model's output as a statistical manifold and follows the geodesic, the shortest path under the Fisher information metric, from a reference distribution to the explained patient [21]. Its Geodesic Sensitivity Field (GSF) scores measure how strongly each variable pushes the prediction along that path; the path length and curvature describe how complex the model is near this patient. It is used here in its simplest form.

In [ ]:
gx = Explainer(gb, data_type="tabular", feature_names=FEATURES, class_names=["no disease", "disease"])
res = gx.explain(x_pat.to_numpy(), X_reference=X_train.to_numpy())
print(res.summary())
res.plot("gsf_bar", theme="light")
plt.show()

## 7. Can the explanation be trusted?

Different methods can give different answers for the same prediction, and some of them may not reflect what the model actually does [22, 23]. Four checks are applied below to the methods that explain the same gradient boosting model (SHAP, LIME, MAPLE, GEMEX and single-variable occlusion), for several test patients predicted to have disease.

**Faithfulness (deletion).** Variables are removed in the order of their attributed importance, replacing them with background values, and the predicted risk is tracked. A faithful explanation makes the risk fall fastest; a lower area under the deletion curve is better [24]. **Monotonicity.** The rank correlation between the size of an attribution and the effect of removing that variable alone. **Stability.** A local Lipschitz estimate: the largest change of the explanation relative to a tiny change of the patient's continuous values [25]. **Disagreement.** The rank correlation between methods. Occlusion scores perfectly on monotonicity by construction, because the check uses the same single-variable removal; it serves as a reference, not as a winner.

In [ ]:
BG = background.to_numpy()[:24]


def removal_effects(x_row):
    """Drop in predicted risk when a single variable is replaced by background values."""
    base = risk(x_row.to_frame().T)[0]
    out = []
    for j in range(len(FEATURES)):
        rows = np.tile(x_row.to_numpy(), (len(BG), 1))
        rows[:, j] = BG[:, j]
        out.append(base - risk(pd.DataFrame(rows, columns=FEATURES)).mean())
    return np.array(out)


def deletion_area(x_row, attribution):
    order = np.argsort(-np.abs(attribution))
    rows = np.tile(x_row.to_numpy(), (len(BG), 1))
    curve = [risk(x_row.to_frame().T)[0]]
    for j in order:
        rows[:, j] = BG[:, j]
        curve.append(risk(pd.DataFrame(rows, columns=FEATURES)).mean())
    return trapezoid(curve, dx=1 / len(order))


def attributions(x_row, gemex_result=None):
    out = {"SHAP": explainer(x_row.to_frame().T).values[0],
           "LIME": lime_attribution(x_row, seed=0),
           "MAPLE": maple_attribution(x_row),
           "Occlusion": removal_effects(x_row)}
    res_ = gemex_result or gx.explain(x_row.to_numpy(), X_reference=X_train.to_numpy())
    out["GEMEX"] = np.asarray(res_.gsf_scores, dtype=float)
    return out


positives = np.where(proba_test >= 0.5)[0]
eval_ids = [i_pat] + list(rng.choice(positives[positives != i_pat], 2 if QUICK else 6, replace=False))
rows_eval = []
for n, i in enumerate(eval_ids):
    x_i = X_test.iloc[i]
    att = attributions(x_i, res if n == 0 else None)
    effect = removal_effects(x_i)
    rand = np.mean([deletion_area(x_i, rng.normal(size=len(FEATURES))) for _ in range(10)])
    for method, a in att.items():
        rows_eval.append({"patient": i, "method": method, "deletion area": deletion_area(x_i, a),
                          "random order": rand, "monotonicity": spearmanr(np.abs(a), np.abs(effect))[0]})
    if n == 0:
        attributions_patient = att
quality = pd.DataFrame(rows_eval).groupby("method")[["deletion area", "random order", "monotonicity"]].mean()
quality.round(3)

In [ ]:
CONT_IDX = [FEATURES.index(c) for c in CONT]


def lipschitz(method, x_row, n_neighbours, seed=SEED):
    rs = np.random.default_rng(seed)
    a0 = attributions_patient[method]
    worst = 0.0
    for _ in range(n_neighbours):
        x_new = x_row.copy()
        x_new.iloc[CONT_IDX] = x_row.iloc[CONT_IDX] + rs.normal(0, 0.05, len(CONT)) * sd.iloc[CONT_IDX]
        if method == "SHAP":
            a1 = explainer(x_new.to_frame().T).values[0]
        elif method == "LIME":
            a1 = lime_attribution(x_new, seed=int(rs.integers(1e6)))
        elif method == "MAPLE":
            a1 = maple_attribution(x_new)
        else:
            a1 = np.asarray(gx.explain(x_new.to_numpy(), X_reference=X_train.to_numpy()).gsf_scores, dtype=float)
        scale_a = np.abs(a0).sum() + 1e-12
        dz = np.linalg.norm(((x_new - x_row) / sd).to_numpy())
        worst = max(worst, np.linalg.norm((a1 - a0) / scale_a) / max(dz, 1e-12))
    return worst


n_nb = 2 if QUICK else 5
stability = pd.Series({m: lipschitz(m, x_pat, n_nb) for m in ["SHAP", "LIME", "MAPLE", "GEMEX"]},
                      name="relative Lipschitz estimate (lower is more stable)")
stability.round(3).to_frame()

In [ ]:
methods = list(attributions_patient)
agree = pd.DataFrame([[spearmanr(np.abs(attributions_patient[a]), np.abs(attributions_patient[b]))[0]
                       for b in methods] for a in methods], index=methods, columns=methods)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(agree.to_numpy(), cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(methods)), methods, rotation=40, ha="right")
ax.set_yticks(range(len(methods)), methods)
for i in range(len(methods)):
    for j in range(len(methods)):
        ax.text(j, i, f"{agree.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Rank agreement between explanation methods")
ax.grid(False)
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

## 8. Interactive what-if analysis

Change the patient's values and watch the risk and its SHAP decomposition update. This is the kind of interaction that the dashboard in NB7 offers to clinicians.

In [ ]:
def what_if(age, resting_bp, cholesterol, max_heart_rate, st_depression, chest_pain_type, exercise_angina,
            major_vessels, thallium_scan):
    x = x_pat.copy()
    x.update(pd.Series(dict(age=age, resting_bp=resting_bp, cholesterol=cholesterol,
                            max_heart_rate=max_heart_rate, st_depression=st_depression,
                            chest_pain_type=chest_pain_type, exercise_angina=exercise_angina,
                            major_vessels=major_vessels, thallium_scan=thallium_scan), dtype=float))
    e = explainer(x.to_frame().T)
    shap.plots.waterfall(e[0], max_display=9, show=False)
    plt.title(f"Predicted risk {risk(x.to_frame().T)[0]:.2f}")
    plt.show()


def slider(name, lo, hi, step):
    return widgets.FloatSlider(value=float(x_pat[name]), min=lo, max=hi, step=step,
                               continuous_update=False, description=name)


widgets.interact(what_if,
                 age=slider("age", 29, 77, 1), resting_bp=slider("resting_bp", 94, 200, 2),
                 cholesterol=slider("cholesterol", 126, 564, 5), max_heart_rate=slider("max_heart_rate", 71, 202, 2),
                 st_depression=slider("st_depression", 0, 6.2, 0.1),
                 chest_pain_type=widgets.Dropdown(options=[1.0, 2.0, 3.0, 4.0], value=float(x_pat["chest_pain_type"])),
                 exercise_angina=widgets.Dropdown(options=[0.0, 1.0], value=float(x_pat["exercise_angina"])),
                 major_vessels=widgets.Dropdown(options=[0.0, 1.0, 2.0, 3.0], value=float(x_pat["major_vessels"])),
                 thallium_scan=widgets.Dropdown(options=[3.0, 6.0, 7.0], value=float(x_pat["thallium_scan"])));

## 9. One decision framework

| If the question is | and the model is | a sound first choice is |
|---|---|---|
| Can clinicians read the whole model? | any tabular task | a glass-box model (logistic regression, GAM, EBM), compared against the black box |
| Which variables drive the model overall? | tree ensemble | TreeSHAP summary, confirmed with permutation importance or SAGE |
| How does risk change with one variable? | variables correlated | ALE rather than PDP; ICE to see heterogeneity |
| Why this prediction? | tree ensemble | TreeSHAP waterfall; LIME only with repeated runs and a stability check |
| Why this prediction? | differentiable network | integrated gradients (image models: Grad-CAM, see NB3) |
| What would change the decision? | any | a constrained counterfactual; an Anchors rule for a sufficient condition |
| Is the explanation trustworthy? | any | deletion faithfulness, stability and agreement with a second method |

## Discussion

1. On this dataset the interpretable models match or exceed the gradient boosting model. Which one should a hospital deploy, and what would change that answer?
2. LIME and SHAP disagree on part of the ranking for the same patient. How should a clinician be told about such disagreement?
3. Which variables in the counterfactual are genuinely actionable, and which only describe the patient's condition?

## Summary

The toolkit spans models that are readable by construction, global and local attribution, contrastive rules, and a geometric view. None of the methods is automatically right. The evaluation section shows how faithfulness, stability and agreement can be measured, and those measurements, rather than the visual appeal of a plot, should decide which explanation reaches a clinician. NB3 carries the same ideas to medical images.

## References

[1] A. Janosi, W. Steinbrunn, M. Pfisterer, and R. Detrano, "Heart Disease," UCI Machine Learning Repository, 1989. doi: 10.24432/C52P4X

[2] R. Detrano et al., "International application of a new probability algorithm for the diagnosis of coronary artery disease," *The American Journal of Cardiology*, vol. 64, no. 5, pp. 304–310, 1989. doi: 10.1016/0002-9149(89)90524-9

[3] C. Rudin, "Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead," *Nature Machine Intelligence*, vol. 1, pp. 206–215, 2019. doi: 10.1038/s42256-019-0048-x

[4] T. Hastie and R. Tibshirani, "Generalized additive models," *Statistical Science*, vol. 1, no. 3, pp. 297–310, 1986. doi: 10.1214/ss/1177013604

[5] Y. Lou, R. Caruana, J. Gehrke, and G. Hooker, "Accurate intelligible models with pairwise interactions," in *Proc. 19th ACM SIGKDD*, 2013, pp. 623–631. doi: 10.1145/2487575.2487579

[6] H. Nori, S. Jenkins, P. Koch, and R. Caruana, "InterpretML: A unified framework for machine learning interpretability," arXiv:1909.09223, 2019.

[7] L. Breiman, "Random forests," *Machine Learning*, vol. 45, pp. 5–32, 2001. doi: 10.1023/A:1010933404324

[8] L. S. Shapley, "A value for n-person games," in *Contributions to the Theory of Games II*, Annals of Mathematics Studies 28, Princeton University Press, 1953, pp. 307–317.

[9] S. M. Lundberg and S.-I. Lee, "A unified approach to interpreting model predictions," in *Advances in Neural Information Processing Systems 30*, 2017, pp. 4765–4774. arXiv:1705.07874

[10] S. M. Lundberg et al., "From local explanations to global understanding with explainable AI for trees," *Nature Machine Intelligence*, vol. 2, pp. 56–67, 2020. doi: 10.1038/s42256-019-0138-9

[11] J. H. Friedman, "Greedy function approximation: A gradient boosting machine," *The Annals of Statistics*, vol. 29, no. 5, pp. 1189–1232, 2001. doi: 10.1214/aos/1013203451

[12] A. Goldstein, A. Kapelner, J. Bleich, and E. Pitkin, "Peeking inside the black box: Visualizing statistical learning with plots of individual conditional expectation," *Journal of Computational and Graphical Statistics*, vol. 24, no. 1, pp. 44–65, 2015. doi: 10.1080/10618600.2014.907095

[13] D. W. Apley and J. Zhu, "Visualizing the effects of predictor variables in black box supervised learning models," *Journal of the Royal Statistical Society Series B*, vol. 82, no. 4, pp. 1059–1086, 2020. doi: 10.1111/rssb.12377

[14] I. Covert, S. M. Lundberg, and S.-I. Lee, "Understanding global feature contributions with additive importance measures," in *Advances in Neural Information Processing Systems 33*, 2020. arXiv:2004.00668

[15] M. T. Ribeiro, S. Singh, and C. Guestrin, "'Why should I trust you?': Explaining the predictions of any classifier," in *Proc. 22nd ACM SIGKDD*, 2016, pp. 1135–1144. doi: 10.1145/2939672.2939778

[16] G. Plumb, D. Molitor, and A. Talwalkar, "Model agnostic supervised local explanations," in *Advances in Neural Information Processing Systems 31*, 2018. arXiv:1807.02910

[17] M. Sundararajan, A. Taly, and Q. Yan, "Axiomatic attribution for deep networks," in *Proc. 34th Int. Conf. Machine Learning (ICML)*, PMLR 70, 2017, pp. 3319–3328. arXiv:1703.01365

[18] S. Wachter, B. Mittelstadt, and C. Russell, "Counterfactual explanations without opening the black box: Automated decisions and the GDPR," *Harvard Journal of Law & Technology*, vol. 31, no. 2, pp. 841–887, 2018. doi: 10.2139/ssrn.3063289

[19] R. K. Mothilal, A. Sharma, and C. Tan, "Explaining machine learning classifiers through diverse counterfactual explanations," in *Proc. Conf. Fairness, Accountability, and Transparency (FAT\*)*, 2020, pp. 607–617. doi: 10.1145/3351095.3372850

[20] M. T. Ribeiro, S. Singh, and C. Guestrin, "Anchors: High-precision model-agnostic explanations," in *Proc. 32nd AAAI Conf. Artificial Intelligence*, 2018, pp. 1527–1535.

[21] U. Kose, "GEMEX: Geodesic Entropic Manifold Explainability," Python package, version 1.2.2, 2026. https://pypi.org/project/gemex/ (paper accepted at IEEE ICHORA 2026)

[22] S. Krishna et al., "The disagreement problem in explainable machine learning: A practitioner's perspective," *Transactions on Machine Learning Research*, 2024. arXiv:2202.01602

[23] J. Adebayo, J. Gilmer, M. Muelly, I. Goodfellow, M. Hardt, and B. Kim, "Sanity checks for saliency maps," in *Advances in Neural Information Processing Systems 31*, 2018. arXiv:1810.03292

[24] W. Samek, A. Binder, G. Montavon, S. Lapuschkin, and K.-R. Müller, "Evaluating the visualization of what a deep neural network has learned," *IEEE Transactions on Neural Networks and Learning Systems*, vol. 28, no. 11, pp. 2660–2673, 2017. doi: 10.1109/TNNLS.2016.2599820

[25] D. Alvarez-Melis and T. S. Jaakkola, "On the robustness of interpretability methods," arXiv:1806.08049, 2018.